# 🧠 TARS HELIX LITE — Быстрый тест (10 мин, A100)

Проверяет что обучение работает корректно.
Чекпоинты автоматически сохраняются на Google Drive.

In [ ]:
#@title 1. 💾 Drive + GPU + Проект
from google.colab import drive
drive.mount('/content/drive')

import torch, os, shutil
assert torch.cuda.is_available(), '❌ GPU не найден!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'✅ GPU: {gpu} ({vram:.0f} GB)')

# Папка для чекпоинтов
DRIVE_SAVE = '/content/drive/MyDrive/TARS/checkpoints'
os.makedirs(DRIVE_SAVE, exist_ok=True)

# Клонировать проект
P = '/content/TarsSSM-Py'
if not os.path.exists(P):
    DRIVE_P = '/content/drive/MyDrive/TARS/TarsSSM-Py'
    if os.path.exists(DRIVE_P):
        !cp -r {DRIVE_P} {P}
        print('📁 Скопировано с Drive')
    else:
        !git clone https://github.com/Vazilll/TarsSSM-Py.git {P}
        print('📥 Клонировано с GitHub')
else:
    print('📁 Проект уже есть')

os.chdir(P)
!pip install -q einops tqdm 2>/dev/null
print(f'📂 {os.getcwd()}')

In [ ]:
#@title 2. 🧪 Smoke Test
!python train_lite.py --test-only

In [ ]:
#@title 3. 📥 Подготовить данные
import os, subprocess, sys

DATA_DIR = '/content/TarsSSM-Py/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Проверить есть ли уже данные
txt_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.txt')] if os.path.exists(DATA_DIR) else []

if txt_files:
    sizes = {f: os.path.getsize(os.path.join(DATA_DIR, f))/1e6 for f in txt_files}
    for f, s in sorted(sizes.items(), key=lambda x: -x[1]):
        print(f'  📄 {f}: {s:.1f} MB')
    print(f'\n✅ {len(txt_files)} файлов, {sum(sizes.values()):.0f} MB')
else:
    # Создать тестовый корпус (русский текст)
    print('📝 Создаю тестовый корпус (русский текст)...')
    corpus = '''Искусственный интеллект — это область компьютерных наук, которая занимается созданием систем, способных выполнять задачи, обычно требующие человеческого интеллекта. Машинное обучение является подмножеством искусственного интеллекта, которое позволяет системам автоматически обучаться и улучшаться на основе опыта без явного программирования.

Нейронные сети — это вычислительные системы, вдохновлённые биологическими нейронными сетями мозга. Глубокое обучение использует многослойные нейронные сети для моделирования сложных паттернов в данных. Трансформеры произвели революцию в обработке естественного языка благодаря механизму внимания.

TARS — это экспериментальная модель на основе архитектуры Mamba-2 и RWKV-7. Она использует State Space Models для эффективной обработки длинных последовательностей с линейной сложностью по памяти. Модель обучается на UTF-8 байтах и использует WSD schedule для оптимизации.

Стоицизм учит нас фокусироваться на том что мы можем контролировать. Дисциплина — это мост между целями и достижениями. Каждый день является возможностью стать лучше. Не жди мотивации — действуй по привычке.

Программирование — это искусство решения проблем. Хороший код должен быть простым, читаемым и поддерживаемым. Python является одним из самых популярных языков для машинного обучения благодаря своей простоте и богатой экосистеме библиотек.
'''
    # Повторить для объёма (~2MB)
    corpus = corpus * 500
    data_path = os.path.join(DATA_DIR, 'test_corpus.txt')
    with open(data_path, 'w', encoding='utf-8') as f:
        f.write(corpus)
    size_mb = os.path.getsize(data_path) / 1e6
    print(f'✅ Создан {data_path}: {size_mb:.1f} MB')

In [ ]:
#@title 4. 🚀 ОБУЧЕНИЕ (~10 мин на A100)
# A100: d=1024, 20 layers, batch=8, bf16
# Small/quick: 1 epoch, 200 steps, seq=512
!python train_lite.py \
    --epochs 1 \
    --steps 200 \
    --seq-len 512 \
    --batch-size 8 \
    --lr 3e-4 \
    --d-model 768 \
    --n-layers 12 \
    --save-dir models/tars_lite

In [ ]:
#@title 5. 💾 Сохранить на Drive + проверка
import shutil, os, torch

DRIVE_SAVE = '/content/drive/MyDrive/TARS/checkpoints'
LOCAL = '/content/TarsSSM-Py/models/tars_lite'

# Копировать чекпоинты
if os.path.exists(LOCAL):
    for f in os.listdir(LOCAL):
        if f.endswith('.pt'):
            src = os.path.join(LOCAL, f)
            shutil.copy2(src, os.path.join(DRIVE_SAVE, f))
            print(f'💾 {f} → Drive ({os.path.getsize(src)/1e6:.0f} MB)')

# Загрузить и проверить
ckpt_path = os.path.join(LOCAL, 'checkpoint_best.pt')
if os.path.exists(ckpt_path):
    import sys; sys.path.insert(0, '/content/TarsSSM-Py')
    from config import TarsConfig
    from brain.mamba2.core.model_lite import TarsHelixLite
    
    ckpt = torch.load(ckpt_path, map_location='cuda')
    print(f'\n✅ Checkpoint OK!')
    print(f'   Epoch: {ckpt.get("epoch", "?")}')
    print(f'   Loss:  {ckpt.get("loss", ckpt.get("best_loss", "?"))}')
    
    # Quick generate test
    cfg_d = ckpt.get('config', {})
    cfg = TarsConfig(**{k:v for k,v in cfg_d.items() if hasattr(TarsConfig,k)})
    model = TarsHelixLite(cfg).cuda()
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.eval()
    prompt = torch.tensor([[208,159,160,168,162,165,210]], device='cuda')  # "Привет" bytes
    with torch.no_grad():
        out = model.generate(prompt, max_new_tokens=50, temperature=0.8)
    print(f'   Generate: {prompt.shape} → {out.shape}')
    print(f'   Output bytes: {out[0,:20].tolist()}...')
else:
    print('❌ Checkpoint не найден!')

print(f'\n📂 Сохранено: {DRIVE_SAVE}')
!ls -lh {DRIVE_SAVE}/